# Population-Level (Run) Analysis

Uses the aggregated output of `X1_data.ipynb` (`run_level_data.parquet`) to
produce the paper's population-level descriptive result (`§What drives belief
revisions?`): estimated marginal means and planned contrasts for **consensus
change** across the four scenarios and two network types.

Same statistical machinery as `X2_agent_analysis.ipynb` (`lme4`/`lmerTest`/`emmeans`
via `rpy2`), but at the run level: each run (not each agent) is one
observation, with `(1 | statement_id) + (1 | graph_id)` as crossed random
effects (network realization is not nested within agent here, since there is
exactly one consensus value per run).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from r_utils import fit_lmer_full
from plot_utils import _configure_fonts, plot_emmeans_grouped_bar

_configure_fonts()


def save_model_results(results_dict: dict, name: str, save_path: Path) -> None:
    """Save a `fit_lmer_full`/`fit_lmer_with_contrasts` results dict to CSV/JSON."""
    import json

    save_path = save_path / name
    save_path.mkdir(parents=True, exist_ok=True)
    for key, df in results_dict.items():
        if isinstance(df, pd.DataFrame):
            df.to_csv(save_path / f"{key}.csv", index=True)
        else:
            with open(save_path / f"{key}.json", "w") as f:
                json.dump(df, f, indent=4)


## Load data

In [ ]:
data_path = Path.cwd().parent / "data" / "outputs" / "aggregated"
result_path = Path.cwd().parent / "data" / "analysis"

agg_path = result_path / "aggregated" / "runs"
cts_path = result_path / "contrasts" / "runs"
agg_path.mkdir(parents=True, exist_ok=True)
cts_path.mkdir(parents=True, exist_ok=True)

dft = pl.read_parquet(data_path / "run_level_data.parquet").to_pandas()

SETTINGS = ["base_llms", "random_roles", "random_experts", "experts"]
dft["setting"] = pd.Categorical(dft["setting"], categories=SETTINGS)
for col in ["graph_type", "statement_id", "graph_seed"]:
    dft[col] = pd.Categorical(dft[col], categories=sorted(dft[col].astype(str).unique()))

dft.head()


## Estimated marginal means (EMMs)

Fit `modal_consensus_change ~ scenario * graph_type + (1 | statement) + (1 | graph_type:graph_seed)`,
plus a graph-type-pooled version.


In [ ]:
COL_RUN_PRETTY_NAMES = {"modal_consensus_change": "Modal Consensus Change (T-0)"}

for var_name, pretty_name in COL_RUN_PRETTY_NAMES.items():
    res = fit_lmer_full(
        df=dft,
        specification=f"{var_name} ~ setting * graph_type + (1 | statement_id) + (1 | graph_type:graph_seed)",
        emm_formula="~setting * graph_type",
        reml=True,
    )
    res_pooled = fit_lmer_full(
        df=dft,
        specification=f"{var_name} ~ setting + (1 | statement_id) + (1 | graph_type:graph_seed)",
        emm_formula="~setting",
        reml=True,
    )
    save_model_results(res, f"{var_name}_setting_graph_type", agg_path)
    save_model_results(res_pooled, f"{var_name}_setting", agg_path)
    print(f"Finished fitting model for {var_name} with setting * graph_type")

    plot_emmeans_grouped_bar(
        emm_df=res["emm"],
        setting_reference=dft,
        y_label=pretty_name,
        title="",
        figsize=(4.5, 3),
        show=True,
        out_fig=agg_path / f"{var_name}_grouped_bar.pdf",
    )


## Contrast analysis

Same four planned contrasts as `X2_agent_analysis.ipynb` (role effect,
specialization effect, role-specialization alignment, composition effect), fit at
the run level for `modal_consensus_change`.

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from typing import Any


def fit_lmer_with_contrasts(
    df: pd.DataFrame,
    outcome: str,
    spec_extra: str = "* graph_type",
    random_effects: str = "(1 | statement_id) + (1 | graph_id)",
    reml: bool = True,
    bootstrap: bool = False,
    n_boot: int = 2000,
    parallel: str = "multicore",
    ncpus: int = 4,
    seed: int = 723522,
) -> dict[str, Any]:
    """Run-level mixed-effects model with `setting` as a fixed effect and four planned contrasts.

    Grouping factors (run level): `(1 | statement_id) + (1 | graph_id)`.
    `graph_id` is the random-effect grouping factor (each distinct graph
    instance); `graph_type` is the fixed within-factor used for the
    within/interaction contrast grids - these are different columns and both
    are intended.

    See `X2_agent_analysis.ipynb`'s `fit_lmer_with_contrasts` for the full
    rationale behind the total-SD Cohen's d and the bootstrap vs. fixed-sigma
    CI construction (identical here, just at the run level).
    """
    specification = f"{outcome} ~ setting {spec_extra} + {random_effects}"
    if "graph_type" not in spec_extra:
        raise ValueError("emmeans grids assume graph_type is in spec_extra")

    with (ro.default_converter + pandas2ri.converter).context():
        ro.globalenv["df"] = ro.conversion.py2rpy(df)
        ro.globalenv["spec"] = specification
        ro.globalenv["use_reml"] = reml
        ro.globalenv["do_boot"] = bootstrap
        ro.globalenv["n_boot"] = n_boot
        ro.globalenv["boot_seed"] = seed
        ro.globalenv["boot_parallel"] = parallel
        ro.globalenv["boot_ncpus"] = ncpus

        ro.r('''
            library(lmerTest); library(emmeans); library(lme4)
            emm_options(pbkrtest.limit = 100000, lmerTest.limit = 100000)

            df$setting <- factor(df$setting, levels = c("base_llms", "random_roles", "random_experts", "experts"))
            df_model <- lmerTest::lmer(as.formula(spec), data = df, REML = use_reml)

            contr_list <- list(
                role_eff     = c(-1, 1, 0, 0),
                model_eff    = c(0, -1, 1, 0),
                alignment    = c(0, 0, -1, 1),
                dissociation = c(-0.5, -0.5, 0.5, 0.5)
            )

            # NOTE: sum(vcov) valid only for intercept-only REs; both REs here
            # (statement_id, graph_id) are intercept-only.
            total_sd <- function(fit) sqrt(sum(as.data.frame(VarCorr(fit))$vcov))
            sigma_total <- total_sd(df_model)

            emm_marginal <- emmeans(df_model, ~ setting, lmer.df = "satterthwaite")
            raw_marginal <- contrast(emm_marginal, contr_list, adjust = "none")
            contrasts_marginal <- as.data.frame(raw_marginal)
            ci_marginal <- as.data.frame(confint(raw_marginal))
            contrasts_marginal$lower.CL <- ci_marginal$lower.CL
            contrasts_marginal$upper.CL <- ci_marginal$upper.CL

            emm_within <- emmeans(df_model, ~ setting | graph_type, lmer.df = "satterthwaite")
            within_obj <- contrast(emm_within, contr_list, adjust = "none")
            contrasts_within <- as.data.frame(within_obj)
            ci_within <- as.data.frame(confint(within_obj))
            contrasts_within$lower.CL <- ci_within$lower.CL
            contrasts_within$upper.CL <- ci_within$upper.CL

            interaction_obj <- pairs(within_obj, by = "contrast", adjust = "none")
            contrasts_interaction <- as.data.frame(interaction_obj)
            ci_interaction <- as.data.frame(confint(interaction_obj))
            contrasts_interaction$lower.CL <- ci_interaction$lower.CL
            contrasts_interaction$upper.CL <- ci_interaction$upper.CL

            n_m <- nrow(contrasts_marginal); n_w <- nrow(contrasts_within); n_i <- nrow(contrasts_interaction)

            contrasts_marginal$cohens_d_total    <- contrasts_marginal$estimate    / sigma_total
            contrasts_within$cohens_d_total      <- contrasts_within$estimate      / sigma_total
            contrasts_interaction$cohens_d_total <- contrasts_interaction$estimate / sigma_total

            boot_diag <- data.frame(n_boot = 0L, n_failed = NA_integer_, frac_failed = NA_real_)

            if (isTRUE(do_boot)) {
                d_total_stat <- function(fit) {
                    em <- emmeans(fit, ~ setting)
                    ew <- emmeans(fit, ~ setting | graph_type)
                    est_m <- as.data.frame(contrast(em, contr_list, adjust = "none"))$estimate
                    wob   <- contrast(ew, contr_list, adjust = "none")
                    est_w <- as.data.frame(wob)$estimate
                    est_i <- as.data.frame(pairs(wob, by = "contrast", adjust = "none"))$estimate
                    c(est_m, est_w, est_i) / total_sd(fit)
                }
                bb <- lme4::bootMer(df_model, d_total_stat, nsim = n_boot, seed = boot_seed,
                                     type = "parametric", parallel = boot_parallel, ncpus = boot_ncpus, use.u = FALSE)

                finite_mask <- apply(bb$t, 1, function(row) all(is.finite(row)))
                n_failed <- sum(!finite_mask)
                bt_clean <- bb$t[finite_mask, , drop = FALSE]
                ci_boot  <- t(apply(bt_clean, 2, quantile, probs = c(.025, .975)))

                idx_m <- 1:n_m; idx_w <- (n_m + 1):(n_m + n_w); idx_i <- (n_m + n_w + 1):(n_m + n_w + n_i)
                contrasts_marginal$cohens_d_total_lower    <- ci_boot[idx_m, 1]
                contrasts_marginal$cohens_d_total_upper    <- ci_boot[idx_m, 2]
                contrasts_within$cohens_d_total_lower      <- ci_boot[idx_w, 1]
                contrasts_within$cohens_d_total_upper      <- ci_boot[idx_w, 2]
                contrasts_interaction$cohens_d_total_lower <- ci_boot[idx_i, 1]
                contrasts_interaction$cohens_d_total_upper <- ci_boot[idx_i, 2]
                boot_diag <- data.frame(n_boot = as.integer(n_boot), n_failed = as.integer(n_failed), frac_failed = n_failed / n_boot)
            } else {
                contrasts_marginal$cohens_d_total_lower    <- contrasts_marginal$lower.CL    / sigma_total
                contrasts_marginal$cohens_d_total_upper    <- contrasts_marginal$upper.CL    / sigma_total
                contrasts_within$cohens_d_total_lower      <- contrasts_within$lower.CL      / sigma_total
                contrasts_within$cohens_d_total_upper      <- contrasts_within$upper.CL      / sigma_total
                contrasts_interaction$cohens_d_total_lower <- contrasts_interaction$lower.CL / sigma_total
                contrasts_interaction$cohens_d_total_upper <- contrasts_interaction$upper.CL / sigma_total
            }

            emm_full <- as.data.frame(emmeans(df_model, ~ setting * graph_type, lmer.df = "satterthwaite"))
            anova_out <- as.data.frame(anova(df_model))
            anova_out$Term <- rownames(anova_out)
            fit_stats <- data.frame(AIC = AIC(df_model), BIC = BIC(df_model), logLik = as.numeric(logLik(df_model)), sigma_total = sigma_total)
        ''')

        return {
            "anova": ro.conversion.rpy2py(ro.r("anova_out")),
            "emm": ro.conversion.rpy2py(ro.r("emm_full")),
            "contrasts_marginal": ro.conversion.rpy2py(ro.r("contrasts_marginal")),
            "contrasts_within": ro.conversion.rpy2py(ro.r("contrasts_within")),
            "contrasts_interaction": ro.conversion.rpy2py(ro.r("contrasts_interaction")),
            "fit_stats": ro.conversion.rpy2py(ro.r("fit_stats")),
            "boot_diag": ro.conversion.rpy2py(ro.r("boot_diag")),
        }


In [ ]:
def build_contrast_summary(col_list: list[str], results_dict: dict) -> pd.DataFrame:
    rows = []
    for col in col_list:
        out = results_dict[col]
        for ctype in ("marginal", "within", "interaction"):
            tbl = out.get(f"contrasts_{ctype}")
            if tbl is None:
                continue
            for _, row in tbl.iterrows():
                p = row.get("p.value", pd.NA)
                rows.append({
                    "outcome": col, "contrast_type": ctype, "graph_type": row.get("graph_type", pd.NA),
                    "contrast": row["contrast"], "estimate": row.get("estimate", pd.NA),
                    "SE": row.get("SE", pd.NA), "t": row.get("t.ratio", pd.NA), "df": row.get("df", pd.NA),
                    "p": p, "lower": row.get("lower.CL", pd.NA), "upper": row.get("upper.CL", pd.NA),
                    "cohens_d": row.get("cohens_d_total", pd.NA),
                    "cohens_d_lower": row.get("cohens_d_total_lower", pd.NA),
                    "cohens_d_upper": row.get("cohens_d_total_upper", pd.NA),
                    "sig": (pd.notna(p) and p < 0.05),
                })
    return pd.DataFrame(rows)


col_list = ["modal_consensus_change"]

results_dict = {
    col: fit_lmer_with_contrasts(dft, outcome=col, spec_extra="* graph_type",
                                  random_effects="(1 | statement_id) + (1 | graph_id)", reml=True)
    for col in col_list
}
summary = build_contrast_summary(col_list, results_dict)
summary.to_csv(cts_path / "contrast_summary.csv", index=False)
summary
